# PX4 Phase 1 Closed-Loop Dynamics PINN Training v13

This notebook trains the first closed-loop Phase 1 dynamics model from the v14 PX4/Gazebo dataset.

Model contract:

```text
x_t, setpoint_t, prev_setpoint_t, dsetpoint_t, dt_s -> dx_t
```

The setpoint is the upper-controller command sent through PX4 offboard position/velocity/yaw interfaces, not raw open-loop thrust.


In [27]:
# Optional in Colab. Safe to skip locally if Drive is already mounted.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', type(exc).__name__, exc)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('torch:', torch.__version__)


device: cuda
torch: 2.10.0+cu128


In [29]:
NOTEBOOK_REVISION = 'v13_closed_loop'
DATASET_GLOB_CANDIDATES = [
    '/content/drive/MyDrive/**/px4_phase1_closed_loop_dynamics_dataset_v5_*',
    '/content/drive/MyDrive/**/processed/px4_phase1_closed_loop_dynamics_dataset_v5_*',
    '/content/**/px4_phase1_closed_loop_dynamics_dataset_v5_*',
]
MANUAL_DATASET_DIR = ''  # Set this to a processed v5 dataset directory if auto-search misses it.

BATCH_SIZE = 2048
EPOCHS = 260
LR = 2.5e-3
WEIGHT_DECAY = 2e-4
GRAD_CLIP = 2.0
DROPOUT = 0.035
HIDDEN = 256
DEPTH = 4

# Soft physics terms. Keep these modest; the closed-loop PX4 plant is not a simple rigid-body-only model.
ALT_KIN_WEIGHT = 0.08
YAW_KIN_WEIGHT = 0.03
RATE_SMOOTH_WEIGHT = 0.01


In [30]:
import glob

def find_dataset_dir():
    if MANUAL_DATASET_DIR:
        p = Path(MANUAL_DATASET_DIR)
        if (p / 'train.csv').exists():
            return p
        raise FileNotFoundError(f'MANUAL_DATASET_DIR does not contain train.csv: {p}')
    matches = []
    for pat in DATASET_GLOB_CANDIDATES:
        matches.extend(Path(p) for p in glob.glob(pat, recursive=True))
    matches = sorted(set(p for p in matches if (p / 'train.csv').exists()))
    if not matches:
        raise FileNotFoundError('No processed v5 closed-loop dataset found. Run px4_dataset_builder_v5_closed_loop first or set MANUAL_DATASET_DIR.')
    return matches[-1]

DATASET_DIR = find_dataset_dir()
print('DATASET_DIR:', DATASET_DIR)
for name in ['dataset_summary.csv', 'filter_report.csv', 'metadata.json']:
    p = DATASET_DIR / name
    print(name, p.exists())


DATASET_DIR: /content/drive/MyDrive/px4_datasets/processed/px4_phase1_closed_loop_dynamics_dataset_v5_20260508_202959
dataset_summary.csv True
filter_report.csv True
metadata.json True


In [31]:
train_df = pd.read_csv(DATASET_DIR / 'train.csv')
val_df = pd.read_csv(DATASET_DIR / 'val.csv')
test_df = pd.read_csv(DATASET_DIR / 'test.csv')
summary_df = pd.read_csv(DATASET_DIR / 'dataset_summary.csv')
print('rows:', {k: len(v) for k, v in [('train', train_df), ('val', val_df), ('test', test_df)]})
display(summary_df)

STATE_COLS = [
    'relative_altitude_m', 'vel_north_m_s', 'vel_east_m_s', 'vel_down_m_s',
    'roll_deg', 'pitch_deg', 'yaw_deg',
    'roll_rate_rad_s', 'pitch_rate_rad_s', 'yaw_rate_rad_s',
]
ACTION_COLS = [
    'ref_north_m', 'ref_east_m', 'ref_down_m',
    'ref_north_m_s', 'ref_east_m_s', 'ref_down_m_s', 'ref_yaw_deg',
]
TARGET_COLS = [f'dx_{c}' for c in STATE_COLS]
print('target columns:', TARGET_COLS)


rows: {'train': 22970, 'val': 4466, 'test': 5359}


,split,scenario,samples
0,test,C00_position_hold_25m,483
1,test,C01_altitude_step_25_28_25m,485
2,test,C02_altitude_step_25_22_25m,464
3,test,C03_north_position_step_pm2m,427
4,test,C04_east_position_step_pm2m,359
5,test,C05_yaw_position_hold_pm20deg,487
6,test,C06_velocity_north_doublet_0p5mps,484
7,test,C07_velocity_east_doublet_0p5mps,438
8,test,C08_vertical_velocity_doublet_0p25mps,492
9,test,C09_mixed_position_sequence,1240


target columns: ['dx_relative_altitude_m', 'dx_vel_north_m_s', 'dx_vel_east_m_s', 'dx_vel_down_m_s', 'dx_roll_deg', 'dx_pitch_deg', 'dx_yaw_deg', 'dx_roll_rate_rad_s', 'dx_pitch_rate_rad_s', 'dx_yaw_rate_rad_s']


In [32]:
def angle_sin_deg(s):
    return np.sin(np.deg2rad(pd.to_numeric(s, errors='coerce').to_numpy(dtype=np.float32)))

def angle_cos_deg(s):
    return np.cos(np.deg2rad(pd.to_numeric(s, errors='coerce').to_numpy(dtype=np.float32)))

def make_features(df):
    parts = []
    names = []

    # Current state. Use sin/cos for yaw to avoid discontinuity at +/-180 deg.
    for col in STATE_COLS:
        src = f'x_{col}'
        if col == 'yaw_deg':
            parts.append(angle_sin_deg(df[src])[:, None]); names.append('x_yaw_sin')
            parts.append(angle_cos_deg(df[src])[:, None]); names.append('x_yaw_cos')
        else:
            parts.append(pd.to_numeric(df[src], errors='coerce').to_numpy(dtype=np.float32)[:, None]); names.append(src)

    # Current, previous, and delta setpoints. Encode yaw references cyclically.
    for prefix in ['u_', 'prev_u_', 'du_']:
        for col in ACTION_COLS:
            src = prefix + col
            if col == 'ref_yaw_deg' and prefix != 'du_':
                parts.append(angle_sin_deg(df[src])[:, None]); names.append(src + '_sin')
                parts.append(angle_cos_deg(df[src])[:, None]); names.append(src + '_cos')
            else:
                values = pd.to_numeric(df[src], errors='coerce').to_numpy(dtype=np.float32)
                if col == 'ref_yaw_deg':
                    values = ((values + 180.0) % 360.0) - 180.0
                parts.append(values[:, None]); names.append(src)

    parts.append(pd.to_numeric(df['dt_s'], errors='coerce').to_numpy(dtype=np.float32)[:, None]); names.append('dt_s')
    X = np.concatenate(parts, axis=1).astype(np.float32)
    return X, names

def make_targets(df):
    Y = np.stack([pd.to_numeric(df[c], errors='coerce').to_numpy(dtype=np.float32) for c in TARGET_COLS], axis=1)
    return Y.astype(np.float32)

X_train, FEATURE_COLS = make_features(train_df)
X_val, _ = make_features(val_df)
X_test, _ = make_features(test_df)
Y_train = make_targets(train_df)
Y_val = make_targets(val_df)
Y_test = make_targets(test_df)

print('feature dim:', X_train.shape[1])
print('target dim:', Y_train.shape[1])
print('bad train finite:', np.isfinite(X_train).all(), np.isfinite(Y_train).all())


feature dim: 35
target dim: 10
bad train finite: True True


In [33]:
class Standardizer:
    def __init__(self, x, eps=1e-6):
        self.mean = torch.tensor(np.nanmean(x, axis=0), dtype=torch.float32)
        self.std = torch.tensor(np.nanstd(x, axis=0), dtype=torch.float32).clamp_min(eps)
    def encode(self, x):
        return (torch.as_tensor(x, dtype=torch.float32) - self.mean) / self.std
    def decode(self, z):
        return z * self.std.to(z.device) + self.mean.to(z.device)
    def to_dict(self):
        return {'mean': self.mean.cpu().numpy().tolist(), 'std': self.std.cpu().numpy().tolist()}

x_scaler = Standardizer(X_train)
y_scaler = Standardizer(Y_train)

def loader_for(X, Y, shuffle):
    Xz = x_scaler.encode(X)
    Yz = y_scaler.encode(Y)
    ds = TensorDataset(Xz, Yz)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False, num_workers=0)

train_loader = loader_for(X_train, Y_train, True)
val_loader = loader_for(X_val, Y_val, False)
test_loader = loader_for(X_test, Y_test, False)


In [34]:
class ResidualMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=256, depth=4, dropout=0.03):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Dropout(dropout)]
            d = hidden
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

model = ResidualMLP(X_train.shape[1], Y_train.shape[1], HIDDEN, DEPTH, DROPOUT).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR * 0.06)

target_index = {name: i for i, name in enumerate(TARGET_COLS)}
feature_index = {name: i for i, name in enumerate(FEATURE_COLS)}


In [35]:
def denorm_x(xz):
    return x_scaler.decode(xz)

def denorm_y(yz):
    return y_scaler.decode(yz)

def physics_losses(xz, pred_yz):
    x = denorm_x(xz)
    dy = denorm_y(pred_yz)
    dt = x[:, feature_index['dt_s']].clamp_min(1e-3)

    # relative_altitude_dot ~= -vel_down in NED coordinates. Use midpoint velocity.
    dx_h = dy[:, target_index['dx_relative_altitude_m']]
    vdown = x[:, feature_index['x_vel_down_m_s']]
    dvdown = dy[:, target_index['dx_vel_down_m_s']]
    vdown_mid = vdown + 0.5 * dvdown
    alt_res = dx_h / dt + vdown_mid
    alt_loss = torch.mean(alt_res ** 2)

    # small-angle yaw kinematic prior; deliberately weak for multicopter closed-loop data.
    dx_yaw_rad = torch.deg2rad(dy[:, target_index['dx_yaw_deg']])
    yaw_rate = x[:, feature_index['x_yaw_rate_rad_s']]
    yaw_res = dx_yaw_rad / dt - yaw_rate
    yaw_loss = torch.mean(yaw_res ** 2)

    rate_cols = ['dx_roll_rate_rad_s', 'dx_pitch_rate_rad_s', 'dx_yaw_rate_rad_s']
    rate_loss = torch.stack([dy[:, target_index[c]].pow(2).mean() for c in rate_cols]).mean()
    return alt_loss, yaw_loss, rate_loss

def run_epoch(loader, train_mode):
    model.train(train_mode)
    totals = {'loss': 0.0, 'data': 0.0, 'alt': 0.0, 'yaw': 0.0, 'rate': 0.0, 'n': 0}
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        if train_mode:
            opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train_mode):
            pred = model(xb)
            data_loss = torch.mean((pred - yb) ** 2)
            alt_loss, yaw_loss, rate_loss = physics_losses(xb, pred)
            loss = data_loss + ALT_KIN_WEIGHT * alt_loss + YAW_KIN_WEIGHT * yaw_loss + RATE_SMOOTH_WEIGHT * rate_loss
            if train_mode:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
        bs = xb.shape[0]
        totals['loss'] += float(loss.detach().cpu()) * bs
        totals['data'] += float(data_loss.detach().cpu()) * bs
        totals['alt'] += float(alt_loss.detach().cpu()) * bs
        totals['yaw'] += float(yaw_loss.detach().cpu()) * bs
        totals['rate'] += float(rate_loss.detach().cpu()) * bs
        totals['n'] += bs
    return {k: v / max(totals['n'], 1) for k, v in totals.items() if k != 'n'}


In [36]:
best_val = float('inf')
best_state = None
history = []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    scheduler.step()
    row = {'epoch': epoch, **{f'train_{k}': v for k, v in tr.items()}, **{f'val_{k}': v for k, v in va.items()}, 'lr': scheduler.get_last_lr()[0]}
    history.append(row)
    if va['loss'] < best_val:
        best_val = va['loss']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch == 1 or epoch % 20 == 0:
        print(f"epoch {epoch:04d} train={tr['loss']:.5f} val={va['loss']:.5f} data={va['data']:.5f} alt={va['alt']:.5f} yaw={va['yaw']:.5f}")

if best_state is not None:
    model.load_state_dict(best_state)
print('best val:', best_val, 'elapsed min:', round((time.time() - t0) / 60, 2))
history_df = pd.DataFrame(history)
display(history_df.tail())


epoch 0001 train=0.66505 val=0.33142 data=0.33041 alt=0.01249 yaw=0.00061
epoch 0020 train=0.10141 val=0.09399 data=0.09355 alt=0.00546 yaw=0.00021
epoch 0040 train=0.08755 val=0.08003 data=0.07962 alt=0.00504 yaw=0.00018
epoch 0060 train=0.08050 val=0.07461 data=0.07420 alt=0.00503 yaw=0.00024
epoch 0080 train=0.07604 val=0.07420 data=0.07375 alt=0.00550 yaw=0.00019
epoch 0100 train=0.07223 val=0.06891 data=0.06847 alt=0.00549 yaw=0.00017
epoch 0120 train=0.06883 val=0.06787 data=0.06739 alt=0.00594 yaw=0.00018
epoch 0140 train=0.06805 val=0.06962 data=0.06917 alt=0.00551 yaw=0.00017
epoch 0160 train=0.06495 val=0.06612 data=0.06566 alt=0.00577 yaw=0.00021
epoch 0180 train=0.06688 val=0.06617 data=0.06569 alt=0.00592 yaw=0.00019
epoch 0200 train=0.06280 val=0.06257 data=0.06210 alt=0.00577 yaw=0.00019
epoch 0220 train=0.06035 val=0.06234 data=0.06188 alt=0.00571 yaw=0.00019
epoch 0240 train=0.05922 val=0.06208 data=0.06162 alt=0.00570 yaw=0.00019
epoch 0260 train=0.05994 val=0.06178 d

,epoch,train_loss,train_data,train_alt,train_yaw,train_rate,val_loss,val_data,val_alt,val_yaw,val_rate,lr
255,256,0.059138,0.058650,0.005922,0.000415,0.000176,0.062368,0.061902,0.005730,0.000190,0.000171,0.000151
256,257,0.058933,0.058451,0.005853,0.000407,0.000175,0.062321,0.061851,0.005783,0.000193,0.000171,0.000151
257,258,0.058803,0.058329,0.005746,0.000413,0.000176,0.062241,0.061772,0.005780,0.000181,0.000169,0.000150
258,259,0.058556,0.058074,0.005853,0.000402,0.000176,0.061855,0.061387,0.005756,0.000195,0.000170,0.000150
259,260,0.059937,0.059462,0.005763,0.000413,0.000174,0.061785,0.061318,0.005738,0.000196,0.000169,0.000150


In [37]:
@torch.no_grad()
def predict_denorm(X):
    model.eval()
    Xz = x_scaler.encode(X).to(device)
    preds = []
    for i in range(0, len(Xz), 8192):
        yz = model(Xz[i:i+8192])
        preds.append(denorm_y(yz).cpu().numpy())
    return np.concatenate(preds, axis=0)

def metric_table(name, X, Y):
    P = predict_denorm(X)
    rows = []
    for i, col in enumerate(TARGET_COLS):
        err = P[:, i] - Y[:, i]
        rows.append({
            'split': name,
            'target': col.replace('dx_', ''),
            'rmse': float(np.sqrt(np.mean(err ** 2))),
            'mae': float(np.mean(np.abs(err))),
            'std_true': float(np.std(Y[:, i])),
        })
    return pd.DataFrame(rows)

metrics_df = pd.concat([
    metric_table('train', X_train, Y_train),
    metric_table('val', X_val, Y_val),
    metric_table('test', X_test, Y_test),
], ignore_index=True)
display(metrics_df)


,split,target,rmse,mae,std_true
0,train,relative_altitude_m,0.002577,0.001470,0.010663
1,train,vel_north_m_s,0.004510,0.002671,0.016575
2,train,vel_east_m_s,0.003740,0.002135,0.013971
3,train,vel_down_m_s,0.004879,0.001826,0.012612
4,train,roll_deg,0.018091,0.007617,0.152376
5,train,pitch_deg,0.021365,0.009442,0.178876
6,train,yaw_deg,0.028457,0.009913,0.218645
7,train,roll_rate_rad_s,0.003034,0.001451,0.014167
8,train,pitch_rate_rad_s,0.002968,0.001545,0.014978
9,train,yaw_rate_rad_s,0.001417,0.000566,0.011198


In [38]:
def rollout_segments(df, max_segments=12, horizon=80):
    out = []
    for (scenario, label), g in df.sort_values(['scenario', 'sample_index']).groupby(['scenario', 'ref_label']):
        idx = g.index.to_numpy()
        if len(idx) < horizon + 1:
            continue
        out.append(g.iloc[:horizon].copy())
        if len(out) >= max_segments:
            break
    return out

@torch.no_grad()
def rollout_error(segment):
    cur = segment.iloc[0].copy()
    pred_states = []
    true_states = []
    for k in range(len(segment)):
        row = segment.iloc[k].copy()
        for c in STATE_COLS:
            row[f'x_{c}'] = cur[f'x_{c}']
        Xk, _ = make_features(pd.DataFrame([row]))
        dx = predict_denorm(Xk)[0]
        next_state = {}
        for i, c in enumerate(STATE_COLS):
            value = float(cur[f'x_{c}'] + dx[i])
            if c == 'yaw_deg':
                value = ((value + 180.0) % 360.0) - 180.0
            next_state[f'x_{c}'] = value
        pred_states.append([next_state[f'x_{c}'] for c in STATE_COLS])
        true_states.append([segment.iloc[k][f'x_next_{c}'] for c in STATE_COLS])
        cur = pd.Series(next_state)
    P = np.asarray(pred_states)
    T = np.asarray(true_states, dtype=np.float32)
    return {
        'scenario': str(segment.iloc[0]['scenario']),
        'ref_label': str(segment.iloc[0]['ref_label']),
        'horizon': len(segment),
        'alt_rmse_m': float(np.sqrt(np.mean((P[:, 0] - T[:, 0]) ** 2))),
        'vn_rmse_m_s': float(np.sqrt(np.mean((P[:, 1] - T[:, 1]) ** 2))),
        've_rmse_m_s': float(np.sqrt(np.mean((P[:, 2] - T[:, 2]) ** 2))),
        'yaw_rmse_deg': float(np.sqrt(np.mean((((P[:, 6] - T[:, 6] + 180) % 360) - 180) ** 2))),
    }

roll_rows = [rollout_error(seg) for seg in rollout_segments(test_df, max_segments=16, horizon=80)]
rollout_df = pd.DataFrame(roll_rows)
display(rollout_df)
print('rollout means:')
display(rollout_df.mean(numeric_only=True).to_frame('mean').T if len(rollout_df) else rollout_df)


,scenario,ref_label,horizon,alt_rmse_m,vn_rmse_m_s,ve_rmse_m_s,yaw_rmse_deg
0,C00_position_hold_25m,hold_25m,80,0.174750,0.048097,0.083355,4.385942
1,C01_altitude_step_25_28_25m,climb_to_28m,80,0.113993,0.030606,0.014106,0.144096
2,C01_altitude_step_25_28_25m,return_25m,80,0.296227,0.026758,0.017466,0.106031
3,C02_altitude_step_25_22_25m,descend_to_22m,80,2.536272,0.939331,0.037144,0.316583
4,C02_altitude_step_25_22_25m,return_25m,80,0.282953,0.017924,0.021200,0.041116
5,C03_north_position_step_pm2m,north_plus_2m,80,0.123196,0.622722,0.046651,0.462052
6,C03_north_position_step_pm2m,recovery,80,0.186906,0.734247,0.038987,0.338484
7,C04_east_position_step_pm2m,recovery,80,0.138737,0.033645,0.215862,0.339147
8,C05_yaw_position_hold_pm20deg,recovery,80,0.113606,0.020272,0.040442,5.781875
9,C05_yaw_position_hold_pm20deg,yaw_minus_20deg,80,0.090633,0.032645,0.039142,3.926865


rollout means:


,horizon,alt_rmse_m,vn_rmse_m_s,ve_rmse_m_s,yaw_rmse_deg
mean,80.0,0.540755,0.21592,0.077825,4.498997


In [39]:
if Path('/content/drive/MyDrive').exists():
    SAVE_ROOT = Path('/content/drive/MyDrive/Colab Result/PINN_MPC/px4_phase1_closed_loop_training_v13')
else:
    SAVE_ROOT = Path('/content/px4_phase1_closed_loop_training_v13')
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
SAVE_DIR = SAVE_ROOT / RUN_STAMP
SAVE_DIR.mkdir(parents=True, exist_ok=True)

ckpt = {
    'revision': NOTEBOOK_REVISION,
    'dataset_dir': str(DATASET_DIR),
    'state_cols': STATE_COLS,
    'action_cols': ACTION_COLS,
    'target_cols': TARGET_COLS,
    'feature_cols': FEATURE_COLS,
    'model_config': {'hidden': HIDDEN, 'depth': DEPTH, 'dropout': DROPOUT},
    'model_state_dict': model.cpu().state_dict(),
    'x_scaler': x_scaler.to_dict(),
    'y_scaler': y_scaler.to_dict(),
    'history': history_df.to_dict(orient='records'),
}
torch.save(ckpt, SAVE_DIR / 'px4_closed_loop_dynamics_pinn_v13.pt')
history_df.to_csv(SAVE_DIR / 'training_history.csv', index=False)
metrics_df.to_csv(SAVE_DIR / 'one_step_metrics.csv', index=False)
rollout_df.to_csv(SAVE_DIR / 'rollout_metrics.csv', index=False)
print('saved:', SAVE_DIR)


saved: /content/drive/MyDrive/Colab Result/PINN_MPC/px4_phase1_closed_loop_training_v13/20260508_113234
